In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, time
import numpy as np, pandas as pd
import torch
from transformers import AutoModel, AutoFeatureExtractor
import librosa

BASE = '/content/drive/MyDrive/standup4ai'
AUDIO_DIR = BASE + '/audio_1000'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'
OUT_DIR = BASE + '/wordlevel_wavlm_features'
CKPT_FILE = BASE + '/wordlevel_checkpoint.json'
os.makedirs(OUT_DIR, exist_ok=True)

# Load checkpoint
done = set()
if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE) as f:
        done = set(json.load(f).get('done', []))

# Build audio lookup
audio_map = {}
for f in os.listdir(AUDIO_DIR):
    if not f.endswith(('.m4a', '.wav', '.mp3')):
        continue
    base = f.rsplit('.', 1)[0]
    if ',' in base:
        base = base.split(',')[0]
    audio_map[base] = os.path.join(AUDIO_DIR, f)

# Build label lookup
label_map = {}
for f in os.listdir(LABEL_DIR):
    if f.endswith('.csv'):
        label_map[f.replace('.csv', '')] = os.path.join(LABEL_DIR, f)

# Videos with both audio and labels, not yet done
overlap = sorted(set(audio_map.keys()) & set(label_map.keys()) - done)
print(f'Audio: {len(audio_map)} | Labels: {len(label_map)} | To process: {len(overlap)}')
print(f'Already done: {len(done)}')


In [ ]:
# Load WavLM on GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Loading WavLM-base...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
SR = 16000
print(f'WavLM ready! Params: {sum(p.numel() for p in wavlm.parameters())/1e6:.1f}M')


In [ ]:
# Fast per-word extraction: load audio ONCE, slice with numpy
def parse_timestamp(ts):
    ts = str(ts).strip().strip('[]')
    p = ts.split(',')
    return float(p[0]), float(p[1])

def extract_video_words(audio_path, word_times, batch_size=64):
    '''
    FAST approach: load entire audio ONCE, slice with numpy.
    This is 10-50x faster than per-word librosa.load with offset/duration.
    '''
    n = len(word_times)
    if n == 0:
        return None

    # Step 1: Load entire audio file ONCE (into memory)
    y_full, _ = librosa.load(audio_path, sr=SR, mono=True)
    n_samples = len(y_full)
    
    # Step 2: Build batch of segments (numpy slices, no re-encoding)
    segments = []
    valid_mask = []
    for t0, t1 in word_times:
        s = int(t0 * SR)
        e = int(t1 * SR)
        e = min(e, n_samples)
        if e > s and (e - s) >= int(0.02 * SR):
            segments.append(y_full[s:e])
            valid_mask.append(True)
        else:
            segments.append(np.zeros(int(0.02 * SR), dtype=np.float32))
            valid_mask.append(False)

    # Step 3: Pad to same length
    max_len = max(len(s) for s in segments)
    padded = np.zeros((len(segments), max_len), dtype=np.float32)
    for i, s in enumerate(segments):
        padded[i, :len(s)] = s

    # Step 4: Batch through WavLM
    all_embeddings = []
    for batch_start in range(0, n, batch_size):
        batch = torch.tensor(padded[batch_start:batch_start+batch_size], dtype=torch.float32).to(device)
        with torch.no_grad():
            out = wavlm(batch).last_hidden_state  # (batch, seq, 768)
            emb = out.mean(dim=1).squeeze(1)      # (batch, 768)
        all_embeddings.append(emb.cpu().numpy())
    
    features = np.vstack(all_embeddings)
    return features

print('Fast extractor ready!')


In [ ]:
# Process all videos
t0 = time.time()
for i, vid in enumerate(overlap):
    out_file = OUT_DIR + '/' + vid + '_word_features.npy'
    if os.path.exists(out_file):
        continue

    df = pd.read_csv(label_map[vid])
    word_times, word_labels = [], []
    for _, row in df.iterrows():
        try:
            t0_w, t1_w = parse_timestamp(row['timestamp'])
            word_times.append((t0_w, t1_w))
            word_labels.append(str(row['label']).strip())
        except:
            continue

    feats = extract_video_words(audio_map[vid], word_times)

    if feats is not None and len(feats) > 0:
        np.save(out_file, feats)
        done.add(vid)

    elapsed = time.time() - t0
    rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
    print(f'{i+1}/{len(overlap)} {vid}: {feats.shape if feats is not None else "FAIL"} | done={len(done)} | {rate:.0f}/hr')

    if len(done) % 10 == 0:
        with open(CKPT_FILE, 'w') as f:
            json.dump({'done': list(done)}, f)

with open(CKPT_FILE, 'w') as f:
    json.dump({'done': list(done)}, f)
print(f'\nDone: {len(done)}/{len(overlap)} videos in {(time.time()-t0)/60:.0f} min')


In [ ]:
# Summary
feat_files = sorted([f for f in os.listdir(OUT_DIR) if f.endswith('_word_features.npy')])
print(f'Word-level features: {len(feat_files)} videos')
for f in feat_files[:5]:
    d = np.load(OUT_DIR + '/' + f)
    print(f'  {f}: {d.shape}')
